# Smart Parking Detection 

> **Goal:** Fine-tune YOLOv8m on a custom parking dataset, track experiments with MLflow, compare models, and export to ONNX.

---

##  Notebook Structure

| # | Cell | Description |
|---|------|-------------|
| 1 | Install | Install all required libraries |
| 2 | Dataset | Load dataset from Kaggle |
| 3 | Baseline | Evaluate YOLOv8m **before** fine-tuning |
| 4 | Fine-Tune | Train YOLOv8m on our dataset |
| 5 | Evaluate | Evaluate **after** fine-tuning + log to MLflow |
| 6 | YOLOv8n | Train smaller model for comparison |
| 7 | Compare | Final comparison table across all models |
| 8 | ONNX | Export best model + speed benchmark |
| 9 | Download | Package all results into a zip file |



---

##  Why This Approach?

- We use **YOLOv8m** (medium) as our main model — best accuracy/speed tradeoff
- We measure performance **before and after** fine-tuning to prove improvement
- We use **MLflow** to log every run automatically — no manual tracking
- We train **YOLOv8n** as well to show we understand model size tradeoffs


---
##  Cell 1 — Install Libraries

### What we're installing:
| Library | Purpose |
|---------|---------|
| `ultralytics` | YOLOv8 — training, evaluation, export |
| `mlflow` | Experiment tracking — logs metrics, params, artifacts |

### Why these versions?
We pin versions to avoid breaking changes between runs.
Kaggle already has most dependencies, so this installs fast.


In [2]:
!pip install ultralytics mlflow -q

# ── Verify installation ───────────────────────────────────────────────────────
import ultralytics
import mlflow
import torch

print(f" Ultralytics : {ultralytics.__version__}")
print(f" MLflow      : {mlflow.__version__}")
print(f" PyTorch     : {torch.__version__}")
print(f" CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f" GPU: {torch.cuda.get_device_name(0)}")
    print(f" VRAM: {round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)} GB")
else:
    print("  No GPU detected — make sure Accelerator is set to GPU P100 in settings")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 100.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 64.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

---
## Cell 2 — Load Dataset from Kaggle

### What this cell does:
1. Points to your dataset already uploaded on Kaggle
2. Verifies the folder structure is correct
3. Creates or validates `data.yaml` — the config file YOLOv8 needs
4. Creates a `dataset` object so all other cells work without changes

### Expected folder structure:
```
/kaggle/input/YOUR-DATASET-NAME/
    ├── train/
    │   ├── images/    ← training images
    │   └── labels/    ← YOLO format .txt files
    ├── valid/
    │   ├── images/
    │   └── labels/
    └── data.yaml      ← might already exist
```

###  Action required:
Change `YOUR-DATASET-NAME` to match your actual Kaggle dataset name.
You can find it in: `kaggle.com/datasets/USERNAME/`**`DATASET-NAME`**


In [3]:
import os
import yaml


DATASET_PATH = "/kaggle/input/datasets/abdomohamed782/car-parking-detection/Car_Parking_Detection.v1i.yolov8 (1)/Car_Parking_Detection.v1i.yolov8 (1)"

# ── Verify ────────────────────────────────────────────────────────
assert os.path.exists(DATASET_PATH), f"❌ Not found: {DATASET_PATH}"

# ── Show folder structure ─────────────────────────────────────────
total_images = 0
for split in ["train", "valid", "test"]:
    split_path = f"{DATASET_PATH}/{split}/images"
    if os.path.exists(split_path):
        n = len(os.listdir(split_path))
        total_images += n
        print(f"   {split:6} → {n} images")
    else:
        print(f"   {split:6} →   not found")
print(f"   TOTAL  → {total_images} images")


yaml_path = f"{DATASET_PATH}/data.yaml"
print(f"\n data.yaml: {yaml_path}")

# ── Create dataset object ─────────────────────────────────────────
class DatasetInfo:
    def __init__(self, location, yaml):
        self.location  = location
        self.yaml_path = yaml

dataset = DatasetInfo(DATASET_PATH, yaml_path)
print(f" Dataset ready!")
print(f"   Location  : {dataset.location}")
print(f"   YAML path : {dataset.yaml_path}")

   train  → 723 images
   valid  → 30 images
   test   → 30 images
   TOTAL  → 783 images

✅ data.yaml: /kaggle/input/datasets/abdomohamed782/car-parking-detection/Car_Parking_Detection.v1i.yolov8 (1)/Car_Parking_Detection.v1i.yolov8 (1)/data.yaml
✅ Dataset ready!
   Location  : /kaggle/input/datasets/abdomohamed782/car-parking-detection/Car_Parking_Detection.v1i.yolov8 (1)/Car_Parking_Detection.v1i.yolov8 (1)
   YAML path : /kaggle/input/datasets/abdomohamed782/car-parking-detection/Car_Parking_Detection.v1i.yolov8 (1)/Car_Parking_Detection.v1i.yolov8 (1)/data.yaml


---
## Cell 3 — Baseline Evaluation (Before Fine-Tuning)

### What this cell does:
Evaluates the **pretrained YOLOv8m** on our parking dataset **before any training**.

### Why is this important?
This is the most important step for your CV project.
Without a baseline, you can only say *"I trained a model"*.
With a baseline, you can say *"I improved mAP50 from 0.71 → 0.91 — a 28% gain"*.

That second sentence is what gets you noticed.

### What gets logged to MLflow:
- `mAP50` — main accuracy metric
- `mAP50_95` — stricter accuracy (averaged over IoU thresholds)
- `precision` — how many detections were correct
- `recall` — how many actual objects were found


In [4]:
from ultralytics import YOLO
import mlflow

# ── MLflow setup ──────────────────────────────────────────────────────────────
# All runs will be grouped under "parking-detection" experiment
mlflow.set_tracking_uri("./mlruns")
mlflow.set_experiment("parking-detection")

# ── Load pretrained YOLOv8m (no fine-tuning yet) ─────────────────────────────
# "yolov8m.pt" = pretrained on ImageNet/COCO, never seen our parking data
print(" Loading pretrained YOLOv8m...")
baseline_model = YOLO("yolov8m.pt")

# ── Evaluate on our parking dataset ──────────────────────────────────────────
print(" Evaluating on parking dataset (before fine-tuning)...")
baseline_metrics = baseline_model.val(
    data=dataset.yaml_path,
    verbose=False,
    plots=False,
)

# ── Extract key metrics ───────────────────────────────────────────────────────
b_map50     = round(float(baseline_metrics.box.map50), 4)
b_map50_95  = round(float(baseline_metrics.box.map),   4)
b_precision = round(float(baseline_metrics.box.mp),    4)
b_recall    = round(float(baseline_metrics.box.mr),    4)

# ── Log to MLflow ─────────────────────────────────────────────────────────────
with mlflow.start_run(run_name="01_baseline_yolov8m_pretrained"):
    mlflow.log_param("model",      "yolov8m")
    mlflow.log_param("fine_tuned", False)
    mlflow.log_param("stage",      "baseline — before fine-tuning")

    mlflow.log_metric("mAP50",     b_map50)
    mlflow.log_metric("mAP50_95",  b_map50_95)
    mlflow.log_metric("precision", b_precision)
    mlflow.log_metric("recall",    b_recall)

# ── Print results ─────────────────────────────────────────────────────────────
print("\n" + "="*45)
print("  BASELINE — before fine-tuning")
print("="*45)
print(f"  mAP50     : {b_map50}")
print(f"  mAP50-95  : {b_map50_95}")
print(f"  Precision : {b_precision}")
print(f"  Recall    : {b_recall}")
print("="*45)
print("\n Saved to MLflow → run: '01_baseline_yolov8m_pretrained'")
print(" Keep these numbers — we compare against them after training")

/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/05/01 11:48:08 INFO mlflow.tracking.fluent: Experiment with name 'parking-detection' does not exist. Creating a new experiment.


🔄 Loading pretrained YOLOv8m...
📊 Evaluating on parking dataset (before fine-tuning)...
Ultralytics 8.4.45 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8m summary (fused): 92 layers, 25,886,080 parameters, 0 gradients, 78.9 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.0 ms, read: 8.4±2.9 MB/s, size: 29.8 KB)
val: Scanning /kaggle/input/datasets/abdomohamed782/car-parking-detection/Car_Parking_Detection.v1i.yolov8 (1)/Car_Parking_Detection.v1i.yolov8 (1)/valid/labels... 30 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 30/30 209.5it/s 0.1s.3s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/abdomohamed782/car-parking-detection/Car_Parking_Detection.v1i.yolov8 (1)/Car_Parking_Detection.v1i.yolov8 (1)/valid is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.1it/s 1.9s5.2s
                   all         30        458     0.0102    0.00127   0.000114   1.14e-05

---
##  Cell 4 — Fine-Tune YOLOv8m

### What this cell does:
Trains YOLOv8m on our parking dataset starting from pretrained weights.
This is called **fine-tuning** — we don't train from scratch,
we adapt a model that already knows how to detect objects.

### Key hyperparameters explained:

| Parameter | Value | Why |
|-----------|-------|-----|
| `epochs` | 100 | Enough to converge, `patience=20` stops early if needed |
| `optimizer` | AdamW | Better than SGD for fine-tuning on small datasets |
| `lr0` | 0.001 | Standard starting learning rate for fine-tuning |
| `cos_lr` | True | Gradually reduces LR — smoother convergence |
| `warmup_epochs` | 3 | Slowly ramps up LR at start — prevents early instability |
| `patience` | 20 | Stops training if no improvement for 20 epochs |
| `augment` | True | Mosaic, mixup, copy-paste — increases effective dataset size |
| `batch` | 16 | Safe for Kaggle P100 (16GB VRAM) |




In [5]:
from ultralytics import YOLO

# ── Load pretrained weights ───────────────────────────────────────────────────
# We start from pretrained weights, NOT from scratch
# This is what makes it "fine-tuning" instead of "training from scratch"
model_m = YOLO("yolov8m.pt")

print(" Starting fine-tuning YOLOv8m...")
print(f"   Dataset  : {dataset.yaml_path}")
print(f"   Epochs   : 100 (early stop at patience=20)")
print(f"   Batch    : 16")
print(f"   Optimizer: AdamW + cosine LR")
print("-" * 45)

results = model_m.train(
    data=dataset.yaml_path,

    # ── Core ─────────────────────────────────────────────────────────────────
    epochs=100,
    imgsz=640,          
    batch=16,          

    # ── Optimizer ────────────────────────────────────────────────────────────
    optimizer="AdamW",  
    lr0=0.001,          
    lrf=0.01,           
    weight_decay=0.0005,
    cos_lr=True,      
    warmup_epochs=3,    
    # ── Augmentation ─────────────────────────────────────────────────────────
    augment=True,      
    degrees=15.0,      
    fliplr=0.5,        
    hsv_v=0.4,         

    # ── Early stopping ────────────────────────────────────────────────────────
    patience=20,       
    # ── Output ───────────────────────────────────────────────────────────────
    save=True,
    plots=True,        
    name="parking_yolov8m",
    project="runs/detect",
    verbose=True,
)

print("\n Fine-tuning complete!")
print(f"   Best weights : runs/detect/parking_yolov8m/weights/best.pt")
print(f"   Last weights : runs/detect/parking_yolov8m/weights/last.pt")

🚀 Starting fine-tuning YOLOv8m...
   Dataset  : /kaggle/input/datasets/abdomohamed782/car-parking-detection/Car_Parking_Detection.v1i.yolov8 (1)/Car_Parking_Detection.v1i.yolov8 (1)/data.yaml
   Epochs   : 100 (early stop at patience=20)
   Batch    : 16
   Optimizer: AdamW + cosine LR
---------------------------------------------
Ultralytics 8.4.45 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/input/datasets/abdomohamed782/car-parking-detection/Car_Parking_Detection.v1i.yolov8 (1)/Car_Parking_Detection.v1i.yolov8 (1)/data.yaml, degrees=15.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4,

2026/05/01 11:48:29 INFO mlflow.tracking.fluent: Experiment with name 'runs/detect' does not exist. Creating a new experiment.
2026/05/01 11:48:29 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.
2026/05/01 11:48:29 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.


MLflow: logging run_id(094c196031f3457a97a53d3968ec29bf) to ./mlruns
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri ./mlruns'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to /kaggle/working/runs/detect/runs/detect/parking_yolov8m
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/100      6.08G      0.823       1.05      1.104         86        640: 100% ━━━━━━━━━━━━ 46/46 1.8it/s 25.4s0.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.1it/s 0.5s
                   all         30        458      0.588      0.676      0.544      0.465

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/100      6.55G     0.5689     0.4397     0.9546         50        640: 100% ━━━━━━━━━━━━ 46/46 2.0it/s 22.9s0.3ss
        

---
## Cell 5 — Evaluate After Fine-Tuning + Log to MLflow

### What this cell does:
1. Loads the best checkpoint saved during training (`best.pt`)
2. Evaluates it on the validation set
3. Compares results against the baseline from Cell 3
4. Logs everything to MLflow — metrics, plots, and weights

### What gets logged to MLflow:
- All 4 metrics (mAP50, mAP50-95, precision, recall)
- `improvement_vs_baseline` — the key number for your CV
- `confusion_matrix.png` — shows which classes get confused
- `PR_curve.png` — precision vs recall tradeoff
- `F1_curve.png` — F1 score across confidence thresholds
- `best.pt` — the trained weights


In [9]:
from ultralytics import YOLO
import mlflow
import os

# ── Load best checkpoint ──────────────────────────────────────────────────────
# best.pt = checkpoint with highest val mAP during training
ft_model = YOLO("/kaggle/working/runs/detect/runs/detect/parking_yolov8m/weights/best.pt")

# ── Evaluate on validation set ────────────────────────────────────────────────
print(" Evaluating fine-tuned YOLOv8m...")
ft_metrics = ft_model.val(
    data=dataset.yaml_path,
    verbose=False,
    plots=True,        
)

# ── Extract metrics ───────────────────────────────────────────────────────────
ft_map50     = round(float(ft_metrics.box.map50), 4)
ft_map50_95  = round(float(ft_metrics.box.map),   4)
ft_precision = round(float(ft_metrics.box.mp),    4)
ft_recall    = round(float(ft_metrics.box.mr),    4)
improvement  = round(ft_map50 - b_map50, 4)
improvement_pct = round((improvement / b_map50) * 100, 1)

# ── Log to MLflow ─────────────────────────────────────────────────────────────
artifacts_dir = "/kaggle/working/runs/detect/runs/detect/parking_yolov8m"

with mlflow.start_run(run_name="02_finetuned_yolov8m"):

    # hyperparameters used
    mlflow.log_param("model",         "yolov8m")
    mlflow.log_param("fine_tuned",    True)
    mlflow.log_param("epochs",        100)
    mlflow.log_param("optimizer",     "AdamW")
    mlflow.log_param("lr0",           0.001)
    mlflow.log_param("batch",         16)
    mlflow.log_param("imgsz",         640)
    mlflow.log_param("cos_lr",        True)
    mlflow.log_param("patience",      20)

    # performance metrics
    mlflow.log_metric("mAP50",                  ft_map50)
    mlflow.log_metric("mAP50_95",               ft_map50_95)
    mlflow.log_metric("precision",              ft_precision)
    mlflow.log_metric("recall",                 ft_recall)
    mlflow.log_metric("improvement_vs_baseline",improvement)

    # artifacts — plots generated automatically by YOLO
    for fname in [
        "confusion_matrix.png",
        "PR_curve.png",
        "F1_curve.png",
        "results.png",
    ]:
        fpath = f"{artifacts_dir}/{fname}"
        if os.path.exists(fpath):
            mlflow.log_artifact(fpath)
            print(f"   Logged artifact: {fname}")

    # log the weights themselves
    mlflow.log_artifact(f"{artifacts_dir}/weights/best.pt")
    print(f"   Logged artifact: best.pt")

# ── Print before / after comparison ──────────────────────────────────────────
print("\n" + "="*52)
print("  BEFORE vs AFTER FINE-TUNING")
print("="*52)
print(f"  {'Metric':<12} {'Before':>10} {'After':>10} {'Delta':>10}")
print("-"*52)
print(f"  {'mAP50':<12} {b_map50:>10} {ft_map50:>10} {'+' if improvement >= 0 else ''}{improvement:>10}")
print(f"  {'mAP50-95':<12} {b_map50_95:>10} {ft_map50_95:>10}")
print(f"  {'Precision':<12} {b_precision:>10} {ft_precision:>10}")
print(f"  {'Recall':<12} {b_recall:>10} {ft_recall:>10}")
print("="*52)
print(f"\n mAP50 improved by {improvement} ({improvement_pct}%)")
print("\n Saved to MLflow → run: '02_finetuned_yolov8m'")

 Evaluating fine-tuned YOLOv8m...
Ultralytics 8.4.45 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,840,918 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 40.4±23.6 MB/s, size: 30.5 KB)
val: Scanning /kaggle/input/datasets/abdomohamed782/car-parking-detection/Car_Parking_Detection.v1i.yolov8 (1)/Car_Parking_Detection.v1i.yolov8 (1)/valid/labels... 30 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 30/30 486.4it/s 0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/abdomohamed782/car-parking-detection/Car_Parking_Detection.v1i.yolov8 (1)/Car_Parking_Detection.v1i.yolov8 (1)/valid is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.3it/s 1.6s2.5s
                   all         30        458      0.959      0.806      0.883      0.772
Speed: 0.8ms preprocess, 30.5ms inference, 0.0ms loss, 5

---
##  Cell 6 — Train YOLOv8n (For Comparison)

### What this cell does:
Trains a smaller, faster model (YOLOv8n) on the same dataset.

### Why train a second model?
This shows you understand **engineering tradeoffs**:
- YOLOv8n → smaller, faster, less accurate → good for edge devices
- YOLOv8m → larger, slower, more accurate → good for servers

In interviews, you can say:
> *"I tested both and chose YOLOv8m for the best accuracy/speed balance"*

### Note: Only 50 epochs
We use 50 epochs here (not 100) because this is for comparison only,
not for production. Saves ~1 hour of compute time.


In [10]:
from ultralytics import YOLO
import mlflow

# ── Train YOLOv8n ─────────────────────────────────────────────────────────────
print(" Training YOLOv8n (comparison model)...")
print("   50 epochs only — this is for speed/accuracy comparison")
print("-" * 45)

model_n = YOLO("yolov8n.pt")

model_n.train(
    data=dataset.yaml_path,
    epochs=50,              
    imgsz=640,
    batch=32,             
    optimizer="AdamW",
    lr0=0.001,
    cos_lr=True,
    patience=15,
    augment=True,
    save=True,
    plots=True,
    name="parking_yolov8n",
    project="runs/detect",
    verbose=False,          
)

# ── Evaluate ──────────────────────────────────────────────────────────────────
print("\n Evaluating fine-tuned YOLOv8n...")
n_metrics = model_n.val(
    data=dataset.yaml_path,
    verbose=False,
)

n_map50     = round(float(n_metrics.box.map50), 4)
n_map50_95  = round(float(n_metrics.box.map),   4)
n_precision = round(float(n_metrics.box.mp),    4)
n_recall    = round(float(n_metrics.box.mr),    4)

# ── Log to MLflow ─────────────────────────────────────────────────────────────
with mlflow.start_run(run_name="03_finetuned_yolov8n"):
    mlflow.log_param("model",      "yolov8n")
    mlflow.log_param("fine_tuned", True)
    mlflow.log_param("epochs",     50)
    mlflow.log_param("optimizer",  "AdamW")
    mlflow.log_param("note",       "comparison model — speed vs accuracy tradeoff")

    mlflow.log_metric("mAP50",     n_map50)
    mlflow.log_metric("mAP50_95",  n_map50_95)
    mlflow.log_metric("precision", n_precision)
    mlflow.log_metric("recall",    n_recall)

print(f"\n YOLOv8n results:")
print(f"   mAP50     : {n_map50}")
print(f"   mAP50-95  : {n_map50_95}")
print(f"   Precision : {n_precision}")
print(f"   Recall    : {n_recall}")
print("\n Saved to MLflow → run: '03_finetuned_yolov8n'")

 Training YOLOv8n (comparison model)...
   50 epochs only — this is for speed/accuracy comparison
---------------------------------------------
Ultralytics 8.4.45 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/input/datasets/abdomohamed782/car-parking-detection/Car_Parking_Detection.v1i.yolov8 (1)/Car_Parking_Detection.v1i.yolov8 (1)/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, l

2026/05/01 12:50:44 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.
2026/05/01 12:50:44 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.


MLflow: logging run_id(e738e2a5e5244da29e85f93747a5697f) to ./mlruns
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri ./mlruns'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to /kaggle/working/runs/detect/runs/detect/parking_yolov8n
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/50      4.73G      1.119       2.06      1.237        385        640: 100% ━━━━━━━━━━━━ 23/23 1.5it/s 15.1s0.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.7it/s 0.4s
                   all         30        458       0.31      0.342      0.335       0.18

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/50      4.74G     0.7326     0.8361     0.9765        472        640: 100% ━━━━━━━━━━━━ 23/23 1.8it/s 12.7s0.4s
           

---
##  Cell 7 — Final Model Comparison Table

### What this cell does:
1. Measures real FPS for each model (not theoretical)
2. Gets actual file sizes
3. Builds a comparison table across all 3 runs
4. Prints a ready-to-paste README table

### Why measure FPS here instead of relying on specs?
Theoretical FPS numbers from papers use different hardware.
Measuring on the same machine gives honest, reproducible numbers
that you can actually defend in an interview.


In [14]:
import pandas as pd
import time
import numpy as np
import os
import mlflow

# ── FPS measurement function ──────────────────────────────────────────────────
def measure_fps(model_path, n_runs=100):
    """Measure real inference FPS on current hardware"""
    from ultralytics import YOLO
    model = YOLO(model_path)
    dummy = np.zeros((640, 640, 3), dtype=np.uint8)
    # warmup — GPU needs a few runs to reach stable speed
    for _ in range(10):
        model(dummy, verbose=False)
    # benchmark
    start = time.time()
    for _ in range(n_runs):
        model(dummy, verbose=False)
    elapsed = time.time() - start
    return round(n_runs / elapsed, 1)

def get_size_mb(path):
    """Get file size in MB"""
    if os.path.exists(path):
        return round(os.path.getsize(path) / 1e6, 1)
    return "N/A"

# ── Measure FPS ───────────────────────────────────────────────────────────────
print("  Measuring FPS for each model (~1 min)...")

fps_m = measure_fps("runs/detect/runs/detect/parking_yolov8m/weights/best.pt")
print(f"   YOLOv8m fine-tuned  : {fps_m} FPS")

fps_n = measure_fps("runs/detect/runs/detect/parking_yolov8n/weights/best.pt")
print(f"   YOLOv8n fine-tuned  : {fps_n} FPS")

# ── File sizes ────────────────────────────────────────────────────────────────
size_m = get_size_mb("/kaggle/working/runs/detect/runs/detect/parking_yolov8m/weights/best.pt")
size_n = get_size_mb("/kaggle/working/runs/detect/runs/detect/parking_yolov8n/weights/best.pt")

# ── Build comparison table ────────────────────────────────────────────────────
comparison = {
    "YOLOv8m pretrained": {
        "mAP50":      b_map50,
        "mAP50-95":   b_map50_95,
        "Precision":  b_precision,
        "Recall":     b_recall,
        "FPS":        fps_m,
        "Size (MB)":  size_m,
        "Status":     "Baseline",
    },
    "YOLOv8n fine-tuned": {
        "mAP50":      n_map50,
        "mAP50-95":   n_map50_95,
        "Precision":  n_precision,
        "Recall":     n_recall,
        "FPS":        fps_n,
        "Size (MB)":  size_n,
        "Status":     "Fast model",
    },
    "YOLOv8m fine-tuned": {
        "mAP50":      ft_map50,
        "mAP50-95":   ft_map50_95,
        "Precision":  ft_precision,
        "Recall":     ft_recall,
        "FPS":        fps_m,
        "Size (MB)":  size_m,
        "Status":     " Best model",
    },
}

df = pd.DataFrame(comparison).T
df.index.name = "Model"

# ── Print table ───────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("  FINAL MODEL COMPARISON")
print("="*70)
print(df.to_string())
print("="*70)

# ── Save ──────────────────────────────────────────────────────────────────────
os.makedirs("results", exist_ok=True)
df.to_csv("results/model_comparison.csv")
print("\n Saved → results/model_comparison.csv")

# ── Log to MLflow ─────────────────────────────────────────────────────────────
with mlflow.start_run(run_name="04_model_comparison_summary"):
    mlflow.log_metric("best_mAP50",        ft_map50)
    mlflow.log_metric("best_fps",          fps_m)
    mlflow.log_metric("fast_model_mAP50",  n_map50)
    mlflow.log_metric("fast_model_fps",    fps_n)
    mlflow.log_metric("total_improvement", round(ft_map50 - b_map50, 4))
    mlflow.log_artifact("results/model_comparison.csv")

# ── Print README-ready table ──────────────────────────────────────────────────
print("\n" + "="*70)
print("   COPY THIS INTO YOUR README:")
print("="*70)
print("\n| Model | mAP50 | Precision | Recall | FPS | Size |")
print("|-------|-------|-----------|--------|-----|------|")
for name, row in comparison.items():
    print(f"| {name} | {row['mAP50']} | {row['Precision']} | {row['Recall']} | {row['FPS']} | {row['Size (MB)']} MB |")
print()

  Measuring FPS for each model (~1 min)...
   YOLOv8m fine-tuned  : 44.8 FPS
   YOLOv8n fine-tuned  : 134.7 FPS

  FINAL MODEL COMPARISON
                     mAP50 mAP50-95 Precision  Recall    FPS Size (MB)       Status
Model                                                                              
YOLOv8m pretrained  0.0001      0.0    0.0102  0.0013   44.8      52.0     Baseline
YOLOv8n fine-tuned  0.8644   0.7584    0.9345  0.8139  134.7       6.2   Fast model
YOLOv8m fine-tuned  0.8825   0.7717    0.9589  0.8064   44.8      52.0   Best model

 Saved → results/model_comparison.csv

   COPY THIS INTO YOUR README:

| Model | mAP50 | Precision | Recall | FPS | Size |
|-------|-------|-----------|--------|-----|------|
| YOLOv8m pretrained | 0.0001 | 0.0102 | 0.0013 | 44.8 | 52.0 MB |
| YOLOv8n fine-tuned | 0.8644 | 0.9345 | 0.8139 | 134.7 | 6.2 MB |
| YOLOv8m fine-tuned | 0.8825 | 0.9589 | 0.8064 | 44.8 | 52.0 MB |



---
## Cell 8 — ONNX Export + Speed Benchmark

### What this cell does:
1. Exports the best model (`best.pt`) to ONNX format
2. Verifies the ONNX model actually runs correctly
3. Benchmarks PyTorch vs ONNX speed

### Why ONNX?
ONNX (Open Neural Network Exchange) makes your model run on **any platform**
without needing PyTorch installed:
- Windows / Mac / Linux 
- Raspberry Pi 
- Docker without CUDA 
- Web browsers (with ONNX.js) 

In production, ONNX is typically **1.3-1.5x faster** than PyTorch on CPU.

### What gets logged to MLflow:
- PyTorch FPS vs ONNX FPS
- Speedup ratio
- The actual `.onnx` file as an artifact


In [19]:
from ultralytics import YOLO
import numpy as np
import time
import os
import mlflow

# ── Export best model to ONNX ─────────────────────────────────────────────────
print(" Exporting YOLOv8m to ONNX...")

export_model = YOLO("runs/detect/runs/detect/parking_yolov8m/weights/best.pt")
export_model.export(
    format="onnx",
    imgsz=640,
    simplify=True,  
    opset=17,       
)

onnx_path = "runs/detect/runs/detect/parking_yolov8m/weights/best.onnx"
print(f" Exported: {onnx_path}")
print(f"   Size: {round(os.path.getsize(onnx_path)/1e6, 1)} MB")

# ── Verify ONNX model works ───────────────────────────────────────────────────
print("\n Verifying ONNX model...")
try:
    import onnxruntime as ort
    session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
    input_name  = session.get_inputs()[0].name
    input_shape = session.get_inputs()[0].shape
    dummy = np.random.rand(1, 3, 640, 640).astype(np.float32)
    output = session.run(None, {input_name: dummy})
    print(f" ONNX verified!")
    print(f"   Input  : {input_name} {input_shape}")
    print(f"   Output : shape {output[0].shape}")
except ImportError:
    print("  onnxruntime not installed — skipping verification")
    print("   Install with: pip install onnxruntime")

# ── Speed benchmark ───────────────────────────────────────────────────────────
print("\n  Speed benchmark: PyTorch vs ONNX (100 runs each)...")
dummy_img = np.zeros((640, 640, 3), dtype=np.uint8)

# PyTorch
pt_model = YOLO("runs/detect/runs/detect/parking_yolov8m/weights/best.pt")
for _ in range(5): pt_model(dummy_img, verbose=False)  
start = time.time()
for _ in range(100): pt_model(dummy_img, verbose=False)
pt_fps = round(100 / (time.time() - start), 1)

# ONNX
onnx_model = YOLO(onnx_path)
for _ in range(5): onnx_model(dummy_img, verbose=False)  
start = time.time()
for _ in range(100): onnx_model(dummy_img, verbose=False)
onnx_fps = round(100 / (time.time() - start), 1)

speedup = round(onnx_fps / pt_fps, 2)

# ── Log to MLflow ─────────────────────────────────────────────────────────────
with mlflow.start_run(run_name="05_onnx_export"):
    mlflow.log_param("format",       "onnx")
    mlflow.log_param("opset",        17)
    mlflow.log_param("simplified",   True)

    mlflow.log_metric("pytorch_fps", pt_fps)
    mlflow.log_metric("onnx_fps",    onnx_fps)
    mlflow.log_metric("speedup",     speedup)

    mlflow.log_artifact(onnx_path)

# ── Print results ─────────────────────────────────────────────────────────────
print("\n" + "="*40)
print("  EXPORT BENCHMARK")
print("="*40)
print(f"  PyTorch FPS : {pt_fps}")
print(f"  ONNX FPS    : {onnx_fps}")
print(f"  Speedup     : {speedup}x faster")
pt_size   = round(os.path.getsize("runs/detect/runs/detect/parking_yolov8m/weights/best.pt")/1e6, 1)
onnx_size = round(os.path.getsize(onnx_path)/1e6, 1)
print(f"  PT size     : {pt_size} MB")
print(f"  ONNX size   : {onnx_size} MB")
print("="*40)
print("\n Saved to MLflow → run: '05_onnx_export'")

 Exporting YOLOv8m to ONNX...
Ultralytics 8.4.45 🚀 Python-3.12.12 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
Model summary (fused): 93 layers, 25,840,918 parameters, 0 gradients, 78.7 GFLOPs

PyTorch: starting from 'runs/detect/runs/detect/parking_yolov8m/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 6, 8400) (49.6 MB)

ONNX: starting export with onnx 1.20.1 opset 17...
ONNX: slimming with onnxslim 0.1.92...
ONNX: export success ✅ 2.9s, saved as 'runs/detect/runs/detect/parking_yolov8m/weights/best.onnx' (98.8 MB)

Export complete (4.4s)
Results saved to /kaggle/working/runs/detect/runs/detect/parking_yolov8m/weights
Predict:         yolo predict task=detect model=runs/detect/runs/detect/parking_yolov8m/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=runs/detect/runs/detect/parking_yolov8m/weights/best.onnx imgsz=640 data=/kaggle/input/datasets/abdomohamed782/car-parking-detection/Car_Parking_Detection.v1i.yolov8 (1)/Car

---
## Cell 9 — Package & Download Results

### What this cell does:
Collects all the important output files into one folder
and zips them for easy download from Kaggle.

### Files in the zip:
| File | Where it goes in your repo |
|------|---------------------------|
| `best.pt` | `weights/best.pt` |
| `best.onnx` | `weights/best.onnx` |
| `confusion_matrix.png` | `results/` |
| `PR_curve.png` | `results/` |
| `F1_curve.png` | `results/` |
| `training_curves.png` | `results/` |
| `model_comparison.csv` | `results/` |



In [22]:
import shutil
import os
import glob

# ── Auto-detect base path ─────────────────────────────────────────────
base_candidates = glob.glob('/kaggle/working/**/parking_yolov8m', recursive=True)

if not base_candidates:
    raise Exception("❌ Model folder not found")

base_path = base_candidates[0]
print(" Found model path:", base_path)

# ── Create output folder ──────────────────────────────────────────────
os.makedirs("final_results", exist_ok=True)

# ── Files list (UPDATED) ──────────────────────────────────────────────
files_to_collect = [
    ("weights/best.pt",        "best.pt"),
    ("weights/best.onnx",      "best.onnx"),
    ("confusion_matrix.png",   "confusion_matrix.png"),
    ("results.png",            "training_curves.png"),
]

print("\n Collecting files...")
print("-" * 50)

collected = 0

# ── Collect model + plots ─────────────────────────────────────────────
for src_rel, dst_name in files_to_collect:
    src = os.path.join(base_path, src_rel)
    dst = os.path.join("final_results", dst_name)

    if os.path.exists(src):
        shutil.copy2(src, dst)
        size = round(os.path.getsize(dst) / 1e6, 2)
        print(f"   {dst_name:<30} ({size} MB)")
        collected += 1

# ── Collect CSV ───────────────────────────────────────────────────────
csv_path = "results/model_comparison.csv"
if os.path.exists(csv_path):
    shutil.copy2(csv_path, "final_results/model_comparison.csv")
    print("   model_comparison.csv added ")
    collected += 1

# ── Create ZIP ────────────────────────────────────────────────────────
zip_name = "parking_detection_results"
shutil.make_archive(zip_name, "zip", "final_results")

zip_size = round(os.path.getsize(f"{zip_name}.zip") / 1e6, 1)

print("-" * 50)
print(f"\n Created: {zip_name}.zip ({zip_size} MB)")
print(f" Collected {collected} files")

# ── Summary ───────────────────────────────────────────────────────────
print("\n" + "="*55)
print(" FINAL SUMMARY")
print("="*55)

print(f"YOLOv8m mAP50 : {ft_map50}")
print(f"YOLOv8n mAP50 : {n_map50}")
print(f"Speed ratio   : {round(onnx_fps/pt_fps,2)}x")

print("="*55)

print("\n Download from Kaggle Output:")
print(f" {zip_name}.zip")

 Found model path: /kaggle/working/runs/detect/runs/detect/parking_yolov8m

--------------------------------------------------
   best.pt                        (52.04 MB)
   best.onnx                      (103.63 MB)
   confusion_matrix.png           (0.11 MB)
   training_curves.png            (0.3 MB)
   model_comparison.csv added 
--------------------------------------------------

 Created: parking_detection_results.zip (134.6 MB)
 Collected 5 files

 FINAL SUMMARY
YOLOv8m mAP50 : 0.8825
YOLOv8n mAP50 : 0.8644
Speed ratio   : 0.97x

 Download from Kaggle Output:
 parking_detection_results.zip
